## Importing Dependencies

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
import joblib
import torch
import torch.nn as nn

## Loading Data

In [2]:
ohlcv = pd.read_csv('OHLCV_Trading_Volume.csv')
order_book = pd.read_csv('Order_Book.csv')

## Aggregating Order Book Data

Since the data for order books has 10 levels, it might be difficult to merge it with OHLCV data. Thus, we aggregate these levels for the following 4 attributes: **Bid Price**, **Bid Quantity**, **Ask Price** and **Ask Quantity**.

In [3]:
agg_order_book = order_book.groupby(['Timestamp', 'Trading Pair']).agg({
    'Bid Price': ['mean', 'max', 'min'],
    'Bid Quantity': ['mean', 'sum'],
    'Ask Price': ['mean', 'max', 'min'],
    'Ask Quantity': ['mean', 'sum']
})

agg_order_book.columns = ['_'.join(col).strip() for col in agg_order_book.columns]

agg_order_book.reset_index(inplace=True)
order_book = agg_order_book.copy()
del agg_order_book

## Feature Engineering

We now add some extra features like the mean changes and rolling statistics for the **Bid Price**, **Bid Quantity**, **Ask Price** and **Ask Quantity** attributes.

In [4]:
order_book['Bid Price Mean Change'] = order_book['Bid Price_mean'].pct_change()
order_book['Bid Quantity Mean Change'] = order_book['Bid Quantity_mean'].pct_change()
order_book['Ask Price Mean Change'] = order_book['Ask Price_mean'].pct_change()
order_book['Ask Quantity Mean Change'] = order_book['Ask Quantity_mean'].pct_change()

In [5]:
order_book = order_book.sort_values(by='Timestamp', ascending=True).reset_index(drop=True)

In [6]:
def calculate_features(group):
    group['Bid Price Mean Change'] = group['Bid Price_mean'].pct_change()
    group['Bid Quantity Mean Change'] = group['Bid Quantity_mean'].pct_change()
    group['Ask Price Mean Change'] = group['Ask Price_mean'].pct_change()
    group['Ask Quantity Mean Change'] = group['Ask Quantity_mean'].pct_change()

    group['Bid Price Rolling Mean'] = group['Bid Price_mean'].rolling(window=5).mean()
    group['Ask Price Rolling Mean'] = group['Ask Price_mean'].rolling(window=5).mean()
    group['Bid Quantity Rolling Sum'] = group['Bid Quantity_sum'].rolling(window=5).sum()
    group['Ask Quantity Rolling Sum'] = group['Ask Quantity_sum'].rolling(window=5).sum()

    group = group.fillna(0)
    return group

In [7]:
order_book = order_book.groupby('Trading Pair').apply(calculate_features).reset_index(drop=True)

/tmp/ipykernel_886878/2283995780.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  order_book = order_book.groupby('Trading Pair').apply(calculate_features).reset_index(drop=True)


## Merging the data

In [8]:
df = pd.merge(ohlcv, order_book, on=['Timestamp', 'Trading Pair'], how='inner')

In [9]:
df = df.set_index('Timestamp')

## Data Normalization

In [10]:
def normalize_pair_data(df, columns):    
    scaler = StandardScaler()
    
    df_normalized = df.groupby('Trading Pair').apply(
        lambda group: group.assign(
            **{col: scaler.fit_transform(group[[col]]) for col in columns}
        )
    )
    
    return df_normalized

In [11]:
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns
df = normalize_pair_data(df, list(numerical_columns))
df = df.reset_index(level='Trading Pair', drop=True)

/tmp/ipykernel_886878/850294643.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_normalized = df.groupby('Trading Pair').apply(


In [12]:
df

,Trading Pair,Open,High,Low,Close,Volume,24-Hour Volume,Bid Price_mean,Bid Price_max,Bid Price_min,...,Ask Quantity_mean,Ask Quantity_sum,Bid Price Mean Change,Bid Quantity Mean Change,Ask Price Mean Change,Ask Quantity Mean Change,Bid Price Rolling Mean,Ask Price Rolling Mean,Bid Quantity Rolling Sum,Ask Quantity Rolling Sum
Timestamp,,,,,,,,,,,,,,,,,,,,,
2024-09-03 18:00:00,1000SATS/USDT,-0.298443,-0.366565,-0.273376,-0.286022,-0.690318,-1.784032,-1.965548,-1.965548,-1.965548,...,-2.155700,-2.155700,-0.004488,-0.006066,-0.004488,-0.015567,-18.952411,-18.952423,-7.772801,-11.005439
2024-09-03 18:30:00,1000SATS/USDT,-0.288882,-0.300534,-0.201361,-0.266943,-0.758955,-1.815830,-1.965548,-1.965548,-1.965548,...,-2.155700,-2.155700,-0.004488,-0.006066,-0.004488,-0.015567,-18.952411,-18.952423,-7.772801,-11.005439
2024-09-03 19:00:00,1000SATS/USDT,-0.274541,-0.272235,-0.191759,-0.286022,-0.534246,-1.807567,-1.965548,-1.965548,-1.965548,...,-2.155685,-2.155685,-0.004488,-0.006066,-0.004488,-0.015522,-18.952411,-18.952423,-7.772801,-11.005439
2024-09-03 19:30:00,1000SATS/USDT,-0.284102,-0.357132,-0.268575,-0.319410,-0.651418,-1.803156,-1.965548,-1.965548,-1.965548,...,-2.678590,-2.678590,-0.004488,-0.006066,-0.004488,-1.534993,-18.952411,-18.952423,-7.772801,-11.005439
2024-09-03 20:00:00,1000SATS/USDT,-0.327126,-0.309967,-0.239769,-0.276482,-0.786621,-1.811064,-1.965548,-1.965548,-1.965548,...,-2.678590,-2.678590,-0.004488,-0.006066,-0.004488,-0.015567,0.027949,0.028048,1.367855,-1.985623
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-10-03 16:00:00,XRP/USDT,-1.684258,-1.680996,-1.864727,-1.901670,1.389906,3.763159,1.613116,1.613116,1.613116,...,2.164828,2.164828,-0.046617,0.310654,-0.046616,0.156216,0.087479,0.087413,0.811599,2.043057
2024-10-03 16:30:00,XRP/USDT,-1.903825,-1.869262,-1.917553,-1.944238,0.570137,3.786368,1.613116,1.613116,1.613116,...,2.171029,2.171029,-0.046617,-3.780496,-0.046616,0.003743,0.087479,0.087413,0.526145,2.084636
2024-10-03 17:00:00,XRP/USDT,-1.943150,-1.706964,-1.996791,-1.708480,0.714505,3.823486,1.444172,1.444172,1.444172,...,1.808011,1.808011,-4.235740,1.184172,-4.235754,-1.442464,0.086751,0.086687,0.413222,2.021342


In [13]:
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns
X = df[numerical_columns]

## Training on Isolation Forest

In [15]:
iso_forest = IsolationForest(contamination=0.01, random_state=42)
iso_forest.fit(X)

IsolationForest(contamination=0.01, random_state=42)

In [16]:
joblib.dump(iso_forest, 'isolation_forest_model.pkl')

['isolation_forest_model.pkl']

## Calculating Isolation Forest scores and prediction

In [17]:
scores = iso_forest.decision_function(X)
predictions = iso_forest.predict(X) 

In [18]:
result_df = df.copy()

In [19]:
result_df['IsolationForest_Score'] = scores
result_df['IsolationForest_Prediction'] = predictions

In [20]:
result_df[['Trading Pair', 'IsolationForest_Score', 'IsolationForest_Prediction']].head(50)

,Trading Pair,IsolationForest_Score,IsolationForest_Prediction
Timestamp,,,
2024-09-03 18:00:00,1000SATS/USDT,-0.032712,-1
2024-09-03 18:30:00,1000SATS/USDT,-0.033940,-1
2024-09-03 19:00:00,1000SATS/USDT,-0.031505,-1
2024-09-03 19:30:00,1000SATS/USDT,-0.069003,-1
2024-09-03 20:00:00,1000SATS/USDT,0.048454,1
2024-09-03 20:30:00,1000SATS/USDT,0.052030,1
2024-09-03 21:00:00,1000SATS/USDT,0.057052,1
2024-09-03 21:30:00,1000SATS/USDT,0.030857,1
2024-09-03 22:00:00,1000SATS/USDT,0.053547,1


## Anomalies per trading pair using Isolation Forest

In [21]:
filtered_df = result_df[result_df['IsolationForest_Prediction'] == -1]

In [22]:
count_per_pair = filtered_df['Trading Pair'].value_counts()

In [23]:
sorted_count = count_per_pair.sort_values(ascending=False)

In [24]:
sorted_count.head(20)

Trading Pair
FET/USDT         57
1000SATS/USDT    56
DOGE/USDT        42
BONK/USDT        26
FLOKI/USDT       26
DOGS/USDT        23
NEAR/USDT        23
PEOPLE/USDT      20
TAO/USDT         19
SOL/USDT         18
WIF/USDT         18
APT/USDT         17
AAVE/USDT        16
DOGE/FDUSD       15
TON/USDT         15
ENA/USDT         15
FTM/USDT         15
BNB/USDT         14
SUI/USDT         14
SEI/USDT         14
Name: count, dtype: int64

## Anomaly Detection with Autoencoder

In [27]:
data = df.drop('Trading Pair', axis=1).values
X_train, X_test = train_test_split(data, test_size=0.2, shuffle=False)

In [28]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

In [29]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Autoencoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


In [30]:
input_dim = X_train.shape[1]
hidden_dim = 64
autoencoder = Autoencoder(input_dim=input_dim, hidden_dim=hidden_dim)

In [31]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001)
epochs = 50 
train_loss = []

In [32]:
for epoch in range(epochs):
    autoencoder.train()
    optimizer.zero_grad()
    output = autoencoder(X_train_tensor)
    loss = criterion(output, X_train_tensor)
    loss.backward()
    optimizer.step()
    
    train_loss.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [10/50], Loss: 1.2193
Epoch [20/50], Loss: 1.1749
Epoch [30/50], Loss: 1.0864
Epoch [40/50], Loss: 0.9828
Epoch [50/50], Loss: 0.9231


In [33]:
autoencoder.eval()

Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=24, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=24, bias=True)
    (3): Sigmoid()
  )
)

## Calculating Reconstruction Error

In [34]:
with torch.no_grad():
    reconstructed_train = autoencoder(X_train_tensor)

train_reconstruction_error = torch.mean((reconstructed_train - X_train_tensor) ** 2, dim=1)
train_reconstruction_error = train_reconstruction_error.numpy()
train_error_mean = np.mean(train_reconstruction_error)
train_error_std = np.std(train_reconstruction_error)

print(f"Mean Reconstruction Error: {train_error_mean:.4f}")
print(f"Standard Deviation of Reconstruction Error: {train_error_std:.4f}")


Mean Reconstruction Error: 0.9181
Standard Deviation of Reconstruction Error: 2.3161


## Anomaly rates for autoencoder

In [ ]:
threshold = train_error_mean + 2 * train_error_std

In [36]:
anomalies = train_reconstruction_error > threshold
anomaly_rate = np.mean(anomalies) * 100
print(f"Anomaly Rate: {anomaly_rate:.2f}%")

Anomaly Rate: 0.96%


In [37]:
torch.save(autoencoder.state_dict(), 'autoencoder_model.pth')